In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader,Dataset

import random
import re
from collections import Counter

import numpy as np
import pandas as pd

# **Data Cleaning + Data Loading**

In [ ]:
def clean_text(text):
  text=text.lower()
  text=re.sub(r"[^a-zA-Z0-9?.!,]+","",text)
  return text

In [ ]:
from google.colab import files
files.upload()

In [ ]:
pairs=[]
with open("fra.txt", encoding='utf-8') as f:
  for line in f:
    eng,fr=line.strip().split("\t")[:2]
    pairs.append((clean_text(fr),clean_text(eng)))

# **Tokenization**

In [ ]:
def tokenize(text):
  return text.split()

# **Vocab Building**

In [ ]:
def build_vocab(sentences,min_freq=2):
  counter=Counter()

  for sent in sentences:
    counter.update(tokenize(sent))

  vocab={
      "<pad>":0,
      "<unk>":1,
      "<start>":2,
      "<end>":3
  }

  idx=4;
  for word,freq in counter.items():
    if freq>=min_freq:
      vocab[word]=idx
      idx+=1

  return vocab

In [ ]:
fr_sentences=[p[0] for p in pairs]
en_sentences=[p[1] for p in pairs]

fr_vocab=build_vocab(fr_sentences)
en_vocab=build_vocab(en_sentences)

# **Numerical Encoding**

In [ ]:
def encode(sentence,vocab):
  tokens=tokenize(sentence)

  return [vocab["<start>"]]+ [vocab.get(word,vocab["<unk>"]) for word in tokens] +[vocab["<end>"]]

# **Padding**

In [ ]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
  src,trg=zip(*batch)

  src_pad=pad_sequence(src,batch_first=True,padding_value=fr_vocab["<pad>"])
  trg_pad=pad_sequence(trg,batch_first=True,padding_value=en_vocab["<pad>"])

  return src_pad,trg_pad

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, pairs, fr_vocab, en_vocab):
        self.pairs = pairs
        self.fr_vocab = fr_vocab
        self.en_vocab = en_vocab

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        fr_sentence, en_sentence = self.pairs[idx]

        src = torch.tensor(encode(fr_sentence, self.fr_vocab))
        trg = torch.tensor(encode(en_sentence, self.en_vocab))

        return src, trg

In [ ]:
dataset=TranslationDataset(pairs,fr_vocab,en_vocab)
train_loader=DataLoader(dataset,batch_size=32,shuffle=True,collate_fn=collate_fn)

# **Model**

In [ ]:
class Encoder(nn.Module):
  def __init__(self,input_dim,embed_dim,hidden_dim):
    super().__init__()

    self.embedding=nn.Embedding(input_dim,embed_dim)
    self.lstm=nn.LSTM(embed_dim,hidden_dim,batch_first=True)

  def forward(self,x):
    embedded=self.embedding(x)
    outputs,(hidden,cell)=self.lstm(embedded)

    return outputs,(hidden,cell)

In [ ]:
class Attention(nn.Module):
  def __init__(self,hidden_dim):
    super().__init__()

    self.attn=nn.Linear(hidden_dim*2,hidden_dim)
    self.v=nn.Linear(hidden_dim,1,bias=False)

  def forward(self,hidden,encoder_outputs):
    hidden=hidden[-1].unsqueeze(1).repeat(1,encoder_outputs.size(1),1)

    energy=torch.tanh(self.attn(torch.cat((hidden,encoder_outputs),dim=2)))

    attention=self.v(energy).squeeze(2)

    return torch.softmax(attention,dim=1)

In [ ]:
class Decoder(nn.Module):
  def __init__(self,output_dim,embed_dim,hidden_dim,attention):
    super().__init__()

    self.embedding=nn.Embedding(output_dim,embed_dim)
    self.lstm=nn.LSTM(embed_dim+hidden_dim,hidden_dim,batch_first=True)
    self.attention=attention
    self.fc=nn.Linear(hidden_dim,output_dim)

  def forward(self,input,hidden,cell,encoder_outputs):
    input=input.unsqueeze(1)

    embedded=self.embedding(input)

    attn_weights=self.attention(hidden,encoder_outputs)
    context=torch.bmm(attn_weights.unsqueeze(1),encoder_outputs)

    lstm_input=torch.cat((embedded,context),dim=2)

    output,(hidden,cell)=self.lstm(lstm_input,(hidden,cell))
    prediction=self.fc(output.squeeze(1))

    return prediction,hidden,cell

In [ ]:
class Seq2Seq(nn.Module):
  def __init__(self,encoder,decoder):
    super().__init__()

    self.encoder=encoder
    self.decoder=decoder

  def forward(self,src,trg):

    batch_size,trg_len=trg.shape
    output_dim=len(en_vocab)

    outputs=torch.zeros(batch_size,trg_len,output_dim)

    input=trg[:,0]

    encoder_outputs, (hidden, cell) = self.encoder(src)
    for t in range(1,trg_len):
      output,hidden,cell=self.decoder(input,hidden,cell,encoder_outputs)

      outputs[:,t]=output

      input=trg[:,t] # teacher forcing

    return outputs

# **Training**

In [ ]:
encoder = Encoder(len(fr_vocab), 256, 512)
attention = Attention(512)
decoder = Decoder(len(en_vocab), 256, 512, attention)

model = Seq2Seq(encoder, decoder)

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(5):
    for src, tgr in train_loader:

        outputs = model(src, tgr)

        outputs = outputs[:, 1:].reshape(-1, outputs.shape[-1])
        tgr = tgr[:, 1:].reshape(-1)

        loss = criterion(outputs, tgr)

        optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1)

        optimizer.step()

    print(f"Epoch {epoch}, Loss: {loss.item()}")

# **Inference**

In [ ]:
 def translate(sentence):
    model.eval()

    src = torch.tensor(encode(sentence, fr_vocab)).unsqueeze(0)

    encoder_outputs, hidden, cell = model.encoder(src)

    input = torch.tensor([en_vocab["<start>"]])

    outputs = []

    for _ in range(20):
        output, hidden, cell = model.decoder(input, hidden, cell, encoder_outputs)

        pred = output.argmax(1).item()

        if pred == en_vocab["<end>"]:
            break

        outputs.append(pred)
        input = torch.tensor([pred])

    return " ".join([inv_en_vocab[i] for i in outputs])